In [2]:
import pandas as pd
import altair as alt

df = pd.read_csv("AllSchool_17_22.csv")

cols = df.columns.str.lower()

def find_col(keyword):
    matches = [c for c in df.columns if keyword in c.lower()]
    return matches[0] if matches else None

selected_cols = [
    find_col('graduat'),
    find_col('low income'),
    find_col('disadvantaged'),
    find_col('math'),
    find_col('read'),
    find_col('coll'),
]

selected_cols = [c for c in selected_cols if c is not None]

print("Using columns:", selected_cols)

for col in selected_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.replace('*', '', regex=False)
        .str.strip()
        .replace(['', 'nan', 'N/A', '--'], None)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

all_corr = []

for year in sorted(df['year'].dropna().unique()):
    df_year = df[df['year'] == year][selected_cols]
    df_year = df_year.dropna(thresh=3)  

    if len(df_year) > 5:
        corr = df_year.corr().reset_index().melt(id_vars='index')
        corr.columns = ['Variable1', 'Variable2', 'Correlation']
        corr['year'] = year
        all_corr.append(corr)

corr_df = pd.concat(all_corr)

corr_df['Variable1'] = corr_df['Variable1'].str.replace('%', '')
corr_df['Variable2'] = corr_df['Variable2'].str.replace('%', '')

year_dropdown = alt.binding_select(options=sorted(corr_df['year'].unique()))
year_select = alt.param(value=sorted(corr_df['year'].unique())[0], bind=year_dropdown)

heatmap = alt.Chart(corr_df).mark_rect().encode(
    x=alt.X('Variable1:N', title="Factors"),
    y=alt.Y('Variable2:N', title="Factors"),
    color=alt.Color('Correlation:Q', scale=alt.Scale(scheme='blueorange')),
    tooltip=[
        'Variable1',
        'Variable2',
        alt.Tooltip('Correlation:Q', format=".2f")
    ]
).add_params(
    year_select
).transform_filter(
    alt.datum.year == year_select
).properties(
    width=600,
    height=600,
    title="Interactive Correlation Heatmap by Year"
)

heatmap

Using columns: ['High School Graduates (#)', 'Low Income#', 'Economically Disadvantaged#', 'Math', 'Reading / Writing', 'Attending Coll./Univ. (#)']


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

2017: In 2017 there is a strong positive relationship between academic performance variables, particularly between Math and Reading/Writing scores, indicating that students who perform well in one subject tend to perform well in the other. Graduation counts and college attendance are also strongly positively correlated, suggesting that higher graduation outcomes are associated with increased college enrollment. Economically disadvantaged students show a strong negative correlation with both test scores and graduation-related outcomes, indicating that higher levels of economic disadvantage are associated with lower academic performance and reduced post-secondary participation. 

2018: 2018 shows a very similar pattern to 2017, with strong positive correlations between Math and Reading/Writing scores and between graduation counts and college attendance. The negative relationship between economic disadvantage and academic performance remains consistent, reinforcing the idea that socioeconomic status plays a significant role in student outcomes. Overall, the stability of these relationships suggests that structural inequalitites in education persisted across these years.

2019: In 2019, the relationships remain consistent, but the negative correlation between economically disadvantaged students and academic performance appears slightly stronger. This suggests that the gap between disadvantageed and non-disadvantaged students may be widening. Positive relationships among graduation, college attendance, and test scores continue to be strong, highlighting the interconnected nature of academic success indicators. 

2020: Continues to show strong positive correlations between academic performance measures, particularly Math and Reading/Writing. However, the negative relationship between economic disadvantage and both test scores and graduation outcomes remains pronounced. This period is notable as it aligns with the COVID-19 pandemic, which may have exacerbated existing inequalities, further impacting disadvantaged student populations. 

2021: The structure of relationships shifts slightly due to the inclusion of low-income data. A strong positive correlation exists between low-income populations and economic disadvantage, as expected. At the same time, both variables show strong negative correlations with Math and Reading/Writing scores, reinforcing the impact of socioeconomic challenges on academic performance. The positive relationship between graduation and college attendance remains strong, indicating that students who graduate are still likely to pursue higher education despite broader challenges.

2022: Mirrors the patterns observed in 2021, with strong positive correlations between low-income status and economic disadvantage, and strong negative correlations between these factors and academic performance metrics. The consistency across years suggests that socioeconomic disparities continue to significantly influence educational outcomes. Academic performance variables remain highly correlated with each other, indicating that core academic skills continue to develop together. 

In [7]:
import pandas as pd
import altair as alt

df = pd.read_csv("AllSchool_17_22.csv")

def find_col(keyword):
    matches = [c for c in df.columns if keyword in c.lower()]
    return matches[0] if matches else None

grad_col = [c for c in df.columns if '% Graduated' in c][0]
low_income_col = [c for c in df.columns if 'Low Income%' in c][0]

for col in [grad_col, low_income_col]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.replace('*', '', regex=False)
        .str.strip()
        .replace(['', 'nan', 'N/A', '--'], None)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_grouped = df.groupby('year')[[grad_col, low_income_col]].mean().reset_index()
df_grouped = df_grouped.sort_values('year').ffill().bfill()

df_grouped = df_grouped.rename(columns={
    grad_col: "Graduation Rate",
    low_income_col: "Low Income %"
})

df_long = df_grouped.melt(id_vars='year', var_name='Metric', value_name='Value')

line_chart = alt.Chart(df_long).mark_line(point=True).encode(
    x=alt.X('year:O', title="Year"),
    y=alt.Y('Value:Q', title="Percentage"),
    color=alt.Color('Metric:N', title="Metric"),
    tooltip=['year', 'Metric', alt.Tooltip('Value:Q', format=".2f")]
).properties(
    width=600,
    height=400,
    title="Graduation Rates vs. Socioeconomic Trends Over Time in Massachusetts"
)

line_chart

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

This time-series line chart shows that graduation rates in Massachusetts remained relatively stable, with a slight upward trend from 2017 to 2022. In contrast, the percentage of low-income students remains fairly constant over time, with only minor fluctuations. This suggests that while socioeconomic conditions didn't drastically improve or worsen, graduation outcomes remained resillient. The stability of graduation rates despite persistent socioeconomic disparities may indicate the presence of support systems or policies that help maintain overall performance, though underlying inequalities in academic achievement still exist. 